In [3]:
import pickle

import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

from tensorflow import keras
import kerasncp as kncp

import os
from typing import Iterable, Dict
import tensorflow as tf
import kerasncp as kncp
from kerasncp.tf import LTCCell, WiredCfcCell
from tensorflow import keras
import numpy as np
from matplotlib.image import imread
from tqdm import tqdm
from PIL import Image
import pandas as pd
import time
from keras_models import generate_ncp_model

2024-07-09 16:50:18.843357: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0


In [4]:

DROPOUT = 0.1

DEFAULT_NCP_SEED = 22222

IMAGE_SHAPE = (144, 256, 3)
IMAGE_SHAPE_CV = (IMAGE_SHAPE[1], IMAGE_SHAPE[0])

batch_size = None
seq_len = 64
augmentation_params = None
single_step = False
no_norm_layer = False
mymodel = generate_ncp_model(seq_len, IMAGE_SHAPE, augmentation_params, batch_size, DEFAULT_NCP_SEED, single_step, no_norm_layer)
decay_rate: float = 0.95
lr: float = 0.01
lr_schedule = keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=lr, decay_steps=500,
                                                            decay_rate=decay_rate, staircase=True)

2024-07-09 16:50:21.485653: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2024-07-09 16:50:21.486698: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2024-07-09 16:50:22.982351: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-07-09 16:50:22.984147: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA A30 computeCapability: 8.0
coreClock: 1.44GHz coreCount: 56 deviceMemorySize: 23.49GiB deviceMemoryBandwidth: 869.04GiB/s
2024-07-09 16:50:22.984164: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.11.0
2024-07-09 16:50:22.985788: I tensorflow/stream_executor/platform/default/dso_loa

In [5]:
optimizer = keras.optimizers.Adam(learning_rate=lr_schedule)

mymodel.compile(optimizer=optimizer, loss="mean_squared_error", metrics=['mse'])
mymodel.summary()

Model: "model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 64, 144, 256, 3)] 0         
_________________________________________________________________
rescaling (Rescaling)        (None, 64, 144, 256, 3)   0         
_________________________________________________________________
time_distributed (TimeDistri (None, 64, 144, 256, 3)   7         
_________________________________________________________________
time_distributed_1 (TimeDist (None, 64, 70, 126, 24)   1824      
_________________________________________________________________
time_distributed_2 (TimeDist (None, 64, 33, 61, 36)    21636     
_________________________________________________________________
time_distributed_3 (TimeDist (None, 64, 15, 29, 48)    43248     
_________________________________________________________________
time_distributed_4 (TimeDist (None, 64, 13, 27, 64)    27712 

In [6]:
def get_output_normalization(root):
    training_output_mean_fn = os.path.join(root, 'stats', 'training_output_means.csv')
    if os.path.exists(training_output_mean_fn):
        print('Loading training data output means from: %s' % training_output_mean_fn)
        output_means = np.genfromtxt(training_output_mean_fn, delimiter=',')
    else:
        output_means = np.zeros(4)

    training_output_std_fn = os.path.join(root, 'stats', 'training_output_stds.csv')
    if os.path.exists(training_output_std_fn):
        print('Loading training data output std from: %s' % training_output_std_fn)
        output_stds = np.genfromtxt(training_output_std_fn, delimiter=',')
    else:
        output_stds = np.ones(4)

    return output_means, output_stds


def load_dataset_multi(root, image_size, seq_len, shift, stride, label_scale):
    file_ending = 'png'
    IMAGE_SHAPE = (144, 256, 3)
    IMAGE_SHAPE_CV = (IMAGE_SHAPE[1], IMAGE_SHAPE[0])

    def sub_to_batch(sub_feature, sub_label):
        sfb = sub_feature.batch(seq_len, drop_remainder=True)
        slb = sub_label.batch(seq_len, drop_remainder=True)
        return tf.data.Dataset.zip((sfb, slb))
        # return sub.batch(seq_len, drop_remainder=True)

    
    datasets = []

    #output_means, output_stds = get_output_normalization(root)

    
    for directory in range(1, 13):
        csv_file_name = root + "/" + str(directory) + '/data_out.csv'
        labels = np.genfromtxt(csv_file_name, delimiter=',', skip_header=1, dtype=np.float32)
        print("labels", labels)
        # if labels.shape[1] == 4:
        #     labels = (labels - output_means) / output_stds
        #     # labels = labels * label_scale
        # elif labels.shape[1] == 5:
        #     labels = (labels[:, 1:] - output_means) / output_stds
        #     # labels = labels[:,1:] * label_scale
        # else:
        #     raise Exception('Wrong size of input data (expected 4, got %d' % labels.shape[1])
    
        labels_dataset = tf.data.Dataset.from_tensor_slices(labels)
        # n_images = len(os.listdir(os.path.join(root, d))) - 1
        n_images = len([fn for fn in os.listdir('./' + root + "/" + str(directory)) if file_ending in fn])
        print(n_images)
        print("no of imgs", n_images)
        # dataset_np = np.empty((n_images, 256, 256, 3), dtype=np.uint8)
        dataset_np = np.empty((n_images, *image_size), dtype=np.uint8)

        for ix in range(n_images):
            # dataset_np[ix] = imread(os.path.join(root, d, '%06d.jpeg' % ix))
            img_file_name = root + "/" + str(directory) +'/Image' + str(ix + 1) + '.'+ file_ending
            img = Image.open(img_file_name)
            img = img.resize(IMAGE_SHAPE_CV)
            # dataset_np[ix] = img[img.height - image_size[0]:, :, :]
            dataset_np[ix] = img

        images_dataset = tf.data.Dataset.from_tensor_slices(dataset_np)
        dataset = tf.data.Dataset.zip((images_dataset, labels_dataset))
        dataset = dataset.window(seq_len, shift=shift, stride=stride, drop_remainder=True).flat_map(sub_to_batch)
        datasets.append(dataset)

    return datasets

def get_dataset_multi(root, image_size, seq_len, shift, stride, validation_ratio, label_scale, extra_data_root=None):
    ds = load_dataset_multi(root, image_size, seq_len, shift, stride, label_scale)
    print('n bags: %d' % len(ds))
    cnt = 0

    for d in ds:
        for (ix, _) in enumerate(d):
            pass
            cnt += ix
    print('n windows: %d' % cnt)

    val_ix = 0

    # val_ix = int(len(ds) * validation_ratio)
    # print('\nval_ix: %d\n' % val_ix)
    # validation_datasets = ds[:val_ix]

    training_datasets = ds[val_ix:]

    # if either dataset has length 0, trying to call flat map raises error that return type is wrong
    # assert len(training_datasets) > 0 and len(validation_datasets) > 0, f"Training or validation dataset has no points!" \
    #                                                                     f"Train dataset len: {len(training_datasets)}" \
    #                                                                     f"Val dataset len: {len(validation_datasets)}"
    training = tf.data.Dataset.from_tensor_slices(training_datasets).flat_map(lambda x: x)
    # validation = tf.data.Dataset.from_tensor_slices(validation_datasets).flat_map(lambda x: x)

    # return training, validation
    return training

In [12]:
def load_val_dataset_multi(root, image_size, seq_len, shift, stride, label_scale):
    file_ending = 'png'
    IMAGE_SHAPE = (144, 256, 3)
    IMAGE_SHAPE_CV = (IMAGE_SHAPE[1], IMAGE_SHAPE[0])

    def sub_to_batch(sub_feature, sub_label):
        sfb = sub_feature.batch(seq_len, drop_remainder=True)
        slb = sub_label.batch(seq_len, drop_remainder=True)
        return tf.data.Dataset.zip((sfb, slb))
        # return sub.batch(seq_len, drop_remainder=True)

    
    datasets = []

    #output_means, output_stds = get_output_normalization(root)

    
    for directory in range(1, 4):
        csv_file_name = f"{root}/{directory}_Test/data_out.csv"
        labels = np.genfromtxt(csv_file_name, delimiter=',', skip_header=1, dtype=np.float32)
        print("labels", labels)
        # if labels.shape[1] == 4:
        #     labels = (labels - output_means) / output_stds
        #     # labels = labels * label_scale
        # elif labels.shape[1] == 5:
        #     labels = (labels[:, 1:] - output_means) / output_stds
        #     # labels = labels[:,1:] * label_scale
        # else:
        #     raise Exception('Wrong size of input data (expected 4, got %d' % labels.shape[1])
    
        labels_dataset = tf.data.Dataset.from_tensor_slices(labels)
        # n_images = len(os.listdir(os.path.join(root, d))) - 1
        n_images = len([fn for fn in os.listdir('./' + root + "/" + str(directory)) if file_ending in fn])
        print(n_images)
        print("no of imgs", n_images)
        # dataset_np = np.empty((n_images, 256, 256, 3), dtype=np.uint8)
        dataset_np = np.empty((n_images, *image_size), dtype=np.uint8)

        for ix in range(n_images):
            # dataset_np[ix] = imread(os.path.join(root, d, '%06d.jpeg' % ix))
            img_file_name = root + "/" + str(directory) +'/Image' + str(ix + 1) + '.'+ file_ending
            img = Image.open(img_file_name)
            img = img.resize(IMAGE_SHAPE_CV)
            # dataset_np[ix] = img[img.height - image_size[0]:, :, :]
            dataset_np[ix] = img

        images_dataset = tf.data.Dataset.from_tensor_slices(dataset_np)
        dataset = tf.data.Dataset.zip((images_dataset, labels_dataset))
        dataset = dataset.window(seq_len, shift=shift, stride=stride, drop_remainder=True).flat_map(sub_to_batch)
        datasets.append(dataset)

    return datasets

def get_val_dataset_multi(root, image_size, seq_len, shift, stride, validation_ratio, label_scale, extra_data_root=None):
    ds = load_val_dataset_multi(root, image_size, seq_len, shift, stride, label_scale)
    print('n bags: %d' % len(ds))
    cnt = 0

    for d in ds:
        for (ix, _) in enumerate(d):
            pass
            cnt += ix
    print('n windows: %d' % cnt)

    val_ix = 0

    # val_ix = int(len(ds) * validation_ratio)
    # print('\nval_ix: %d\n' % val_ix)
    # validation_datasets = ds[:val_ix]

    training_datasets = ds[val_ix:]

    # if either dataset has length 0, trying to call flat map raises error that return type is wrong
    # assert len(training_datasets) > 0 and len(validation_datasets) > 0, f"Training or validation dataset has no points!" \
    #                                                                     f"Train dataset len: {len(training_datasets)}" \
    #                                                                     f"Val dataset len: {len(validation_datasets)}"
    training = tf.data.Dataset.from_tensor_slices(training_datasets).flat_map(lambda x: x)
    # validation = tf.data.Dataset.from_tensor_slices(validation_datasets).flat_map(lambda x: x)

    # return training, validation
    return training

In [9]:
shift: int = 1
stride: int = 1
decay_rate: float = 0.95
val_split: float = 0.2
label_scale: float = 1
seq_len = 64
val_split: float = 0.3
label_scale: float = 1

#datasets = load_dataset_multi('Test1', IMAGE_SHAPE, seq_len, shift, stride, label_scale)
training_dataset = get_dataset_multi('dataset', IMAGE_SHAPE, seq_len, shift, stride, val_split, label_scale, extra_data_root=None)
print('load dataset shape', training_dataset.element_spec)

training_dataset = training_dataset.batch(64)
print('load dataset shape', training_dataset.element_spec)




labels [[-0.545209    0.5563235  -0.9627457   0.5512168 ]
 [-0.5254169   0.5640678  -0.93462795  0.53914636]
 [-0.4895641   0.5632284  -0.9060196   0.525603  ]
 ...
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]]
640
no of imgs 640
labels [[-0.6370456   0.38259566 -1.0559112   0.5936858 ]
 [-0.6370456   0.38259566 -1.0559112   0.5936858 ]
 [-0.6230051   0.3925644  -1.0348189   0.5833566 ]
 ...
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]]
576
no of imgs 576
labels [[-0.55643934  0.24774413 -0.8744291   0.5117653 ]
 [-0.55643934  0.24774413 -0.8744291   0.5117653 ]
 [-0.5366164   0.25543112 -0.85399973  0.50109065]
 ...
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]]
512
n

2024-07-09 16:57:52.672871: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)
2024-07-09 16:57:52.692419: I tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 3000260000 Hz


n windows: 2602931
load dataset shape (TensorSpec(shape=(64, 144, 256, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(64, 4), dtype=tf.float32, name=None))
load dataset shape (TensorSpec(shape=(None, 64, 144, 256, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(None, 64, 4), dtype=tf.float32, name=None))
labels [[-0.545209    0.5563235  -0.9627457   0.5512168 ]
 [-0.5254169   0.5640678  -0.93462795  0.53914636]
 [-0.4895641   0.5632284  -0.9060196   0.525603  ]
 ...
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]]
640
no of imgs 640
labels [[-0.6370456   0.38259566 -1.0559112   0.5936858 ]
 [-0.6370456   0.38259566 -1.0559112   0.5936858 ]
 [-0.6230051   0.3925644  -1.0348189   0.5833566 ]
 ...
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]]
576
no of imgs 576
labels [[-0.5

KeyboardInterrupt: 

In [13]:

val_data = get_val_dataset_multi('dataset', IMAGE_SHAPE, seq_len, shift, stride, val_split, label_scale, extra_data_root=None)
val_dataset = val_data.batch(64)
print('load val dataset shape', training_dataset.element_spec)

labels [[-5.41797996e-01 -2.48590797e-01 -3.61648440e-01  1.13702190e+00]
 [-5.41797996e-01 -2.48590797e-01 -3.61648440e-01  1.13702190e+00]
 [-5.41684628e-01 -2.48546466e-01 -3.61284882e-01  1.13693190e+00]
 ...
 [-1.10089836e-07 -2.62509957e-07  1.77635684e-15 -0.00000000e+00]
 [-1.10089836e-07 -2.62509957e-07  1.77635684e-15 -0.00000000e+00]
 [-1.07185507e-07 -2.30279298e-07  1.77635684e-15 -0.00000000e+00]]
640
no of imgs 640
labels [[ 1.2867007e-01 -4.6787786e-01 -6.4619792e-01  1.1176184e+00]
 [ 1.1656361e-01 -4.7288632e-01 -6.2885666e-01  1.1005259e+00]
 [ 1.1656361e-01 -4.7288632e-01 -6.2885666e-01  1.1005259e+00]
 ...
 [-7.2703422e-08  1.0099594e-07  1.7763568e-15  1.3322677e-16]
 [-7.2703422e-08  1.0099594e-07  1.7763568e-15  1.3322677e-16]
 [-7.2690824e-08  1.0099595e-07  1.9539926e-15  1.3322677e-16]]
576
no of imgs 576
labels [[ 2.10227892e-01  4.70751911e-01 -7.21448362e-01  1.30846465e+00]
 [ 2.26428017e-01  4.71748769e-01 -7.05450714e-01  1.29748678e+00]
 [ 2.48287156e-

In [ ]:
from IPython.display import display
# To display a specific element from the dataset
for batch in val_dataset:
    images, labels = batch
    # Display the first image and label in the batch
    display(images[0].numpy())
    display(labels[0].numpy())
    print(len(labels[0].numpy()))
    break  # Remove break if you want to see more batches
